In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

BLUE, GREEN, YELLOW, RED, BOLD, RESET = "\033[94m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"


def step(msg):
    print(f"\n{BOLD}{BLUE}▶ {msg}{RESET}")


def done(msg):
    print(f"{GREEN}  ✓ {msg}{RESET}")


def note(msg):
    print(f"{YELLOW}  • {msg}{RESET}")


ROOT = Path.cwd()
while not (ROOT / "data" / "master_listing.parquet").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MASTER = ROOT / "data" / "master_listing.parquet"
FUNNEL = ROOT / "data" / "feature_funnel_numerical.csv"   # written by 03_feature_funnel_numerical
OUT = ROOT / "data" / "correlations_numerical.csv"

MIN_COMPLETENESS = 70     # keep variables strictly above this % complete
METHOD = "spearman"       # rank-based, robust to skewed rents and fees; "pearson" runs in ~5 s
REDUNDANT = 0.9           # |correlation| at or above this flags a redundant pair

step(f"Reading {FUNNEL.name}")
funnel = pd.read_csv(FUNNEL)
done(f"{len(funnel)} numerical variables")

step(f"Keeping variables more than {MIN_COMPLETENESS}% complete")
keep = funnel[funnel["completeness_pct"] > MIN_COMPLETENESS]
done(f"{len(keep)} kept, {len(funnel) - len(keep)} dropped")
for role, n in keep["role"].value_counts().items():
    note(f"{role:9s}{n:>4}")

step(f"Loading {len(keep)} columns from {MASTER.name}")
df = pd.read_parquet(MASTER, columns=keep["variable"].tolist())
done(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
constant = [c for c in df if df[c].nunique() <= 1]
if constant:
    # A column with one value has no variance, so its correlation is undefined.
    df = df.drop(columns=constant)
    print(f"{RED}  ✗ dropped {len(constant)} constant column(s): {constant}{RESET}")

step(f"Computing {METHOD} correlations, every variable vs every variable (~3 min for spearman)")
# Pairwise: each pair uses the rows where both variables are present.
corr = df.corr(method=METHOD)
corr.index.name = "variable"
done(f"{corr.shape[0]} x {corr.shape[1]} matrix")

step(f"Writing {OUT.relative_to(ROOT)}")
corr.round(4).to_csv(OUT)
done(f"{corr.shape[0]} rows x {corr.shape[1]} columns")

step(f"Redundant pairs (|{METHOD}| >= {REDUNDANT})")
role = keep.set_index("variable")["role"]
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().dropna()
         .rename("corr").rename_axis(["var_1", "var_2"]).reset_index())
pairs["roles"] = pairs["var_1"].map(role) + " / " + pairs["var_2"].map(role)
pairs = pairs.reindex(pairs["corr"].abs().sort_values(ascending=False).index).reset_index(drop=True)
redundant = pairs[pairs["corr"].abs() >= REDUNDANT]
is_x = redundant["roles"].isin(["feature / feature", "feature / inferred", "inferred / feature"])
done(f"{len(redundant)} of {len(pairs):,} pairs, {is_x.sum()} of them between two usable features")
for r in redundant[is_x].itertuples():
    print(f"{RED}  ✗ {r.corr:+.3f}{RESET}  {r.var_1}  ~  {r.var_2}")
note(f"the other {(~is_x).sum()} involve leaky / label / latent columns, which are not used as X")
pairs.head(20)


▶ Reading feature_funnel_numerical.csv
  ✓ 108 numerical variables

▶ Keeping variables more than 70% complete
  ✓ 67 kept, 41 dropped
  • feature    37
  • leaky      14
  • latent      8
  • label       7
  • inferred    1

▶ Loading 67 columns from master_listing.parquet


  ✓ 967,661 rows x 67 columns


  ✗ dropped 1 constant column(s): ['truth_lease_event__inferred_term_months']

▶ Computing spearman correlations, every variable vs every variable (~3 min for spearman)


  ✓ 66 x 66 matrix

▶ Writing data/correlations_numerical.csv
  ✓ 66 rows x 66 columns

▶ Redundant pairs (|spearman| >= 0.9)
  ✓ 122 of 2,145 pairs, 15 of them between two usable features
  ✗ +0.976  market_price_observation__day_weighted_concession  ~  market_price_observation__share_days_with_concession
  ✗ +0.969  market_price_observation__n_windows  ~  market_price_observation__n_price_changes
  ✗ +0.967  market_price_observation__max_concession  ~  market_price_observation__day_weighted_concession
  ✗ +0.929  market_building__positive_staff_and_management  ~  market_building__positive_cleanliness_and_maintenance
  ✗ +0.923  market_unit__bed  ~  market_unit__sqft
  ✗ +0.920  market_building__positive_staff_and_management  ~  market_building__positive_amenities_and_features
  ✗ +0.920  market_building__positive_amenities_and_features  ~  market_building__positive_cleanliness_and_maintenance
  ✗ +0.919  market_price_observation__max_concession  ~  market_price_observation__share_day

,var_1,var_2,corr,roles
0,market_price_observation__last_effective_rent,market_price_observation__leased_effective_rent,1.000000,leaky / leaky
1,market_price_observation__last_asking_rent,market_price_observation__leased_asking_rent,1.000000,leaky / leaky
2,market_building__number_stories,truth_building_params__number_stories_true,1.000000,feature / latent
3,market_building__year_built,truth_building_params__year_built_true,1.000000,feature / latent
4,market_building__number_units,truth_building_params__number_units_true,1.000000,feature / latent
5,market_price_observation__last_asking_rent_psf,market_price_observation__leased_asking_rent_psf,1.000000,leaky / leaky
6,truth_lease_event__true_rent,market_price_observation__last_asking_rent,0.999773,label / leaky
7,truth_lease_event__true_rent,market_price_observation__leased_asking_rent,0.999773,label / leaky
8,truth_lease_event__true_effective_rent,market_price_observation__leased_effective_rent,0.999361,label / leaky
9,truth_lease_event__true_effective_rent,market_price_observation__last_effective_rent,0.999361,label / leaky


In [2]:
from datetime import date

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

REPORT = ROOT / "eda" / "reports" / "correlations_numerical.xlsx"
NEG, MID, POS = "2166AC", "F7F7F7", "B2182B"   # blue = negative, white = none, red = positive
FONT = "Arial"
USABLE = {"feature", "inferred"}
ROLE_COLOR = {"feature": "000000", "inferred": "000000", "label": "B45F06", "leaky": "C00000", "latent": "7F7F7F"}
HEADER_FILL = PatternFill("solid", fgColor="D9D9D9")


def blend(v):
    """Hex color for a correlation: white (0) shading to blue (-1) or red (+1)."""
    end = POS if v >= 0 else NEG
    t = min(abs(v), 1)
    return "".join(f"{round(int(MID[i:i + 2], 16) * (1 - t) + int(end[i:i + 2], 16) * t):02X}" for i in (0, 2, 4))


def heatmap(cells):
    """Fill each cell with its color. Fills are written into the file rather than as Excel
    conditional formatting, so they also show in Numbers, Google Sheets and file previews.
    Dark cells get white text so the number stays readable; bold marks a redundant pair."""
    for cell in cells:
        v = cell.value
        cell.fill = PatternFill("solid", fgColor=blend(v))
        cell.font = Font(name=FONT, bold=abs(v) >= REDUNDANT, color="FFFFFF" if abs(v) >= 0.6 else "000000")


def header(ws, row=1):
    for cell in ws[row]:
        cell.font = Font(name=FONT, bold=True, color=cell.font.color)
        cell.fill = HEADER_FILL


wb = Workbook()
readme = wb.active
readme.title = "Read me"
matrix = wb.create_sheet("Matrix")
pairs_ws = wb.create_sheet("Pairs")

step("Matrix sheet: every variable vs every variable")
names = corr.columns.tolist()
n = len(names)
matrix.append(["variable", "role"] + names)
for v in names:
    matrix.append([v, role[v]] + corr.loc[v].round(4).tolist())
for j, cell in enumerate(matrix[1], 1):
    if j > 2:
        cell.font = Font(name=FONT, color=ROLE_COLOR[role[cell.value]])
        cell.alignment = Alignment(text_rotation=90, horizontal="center")
header(matrix)
for r in range(2, n + 2):
    for c in (1, 2):
        matrix.cell(r, c).font = Font(name=FONT, color=ROLE_COLOR[matrix.cell(r, 2).value])
    for c in range(3, n + 3):
        cell = matrix.cell(r, c)
        cell.font, cell.number_format, cell.alignment = Font(name=FONT), "0.00", Alignment(horizontal="center")
heatmap(c for row in matrix.iter_rows(min_row=2, min_col=3) for c in row)
matrix.row_dimensions[1].height = 290
matrix.column_dimensions["A"].width = max(map(len, names)) + 2
matrix.column_dimensions["B"].width = 9
for c in range(3, n + 3):
    matrix.column_dimensions[get_column_letter(c)].width = 5.5
matrix.freeze_panes = "C2"
done(f"{n} x {n} heatmap")

step("Pairs sheet: each pair once, strongest first")
p = pairs.assign(role_1=pairs["var_1"].map(role), role_2=pairs["var_2"].map(role), abs_corr=pairs["corr"].abs())
p["flag"] = np.where(p["abs_corr"] < REDUNDANT, "",
                     np.where(p["role_1"].isin(USABLE) & p["role_2"].isin(USABLE),
                              "redundant: keep one", "involves leaky / label / latent"))
pairs_ws.append(["variable_1", "role_1", "variable_2", "role_2", METHOD, f"|{METHOD}|", "flag"])
for r in p.itertuples():
    pairs_ws.append([r.var_1, r.role_1, r.var_2, r.role_2, round(r.corr, 4), round(r.abs_corr, 4), r.flag])
header(pairs_ws)
for row in pairs_ws.iter_rows(min_row=2):
    keep_one = row[6].value == "redundant: keep one"
    for cell in row:
        cell.font = Font(name=FONT, bold=keep_one)
    row[4].number_format = row[5].number_format = "0.000"
    if keep_one:
        row[6].fill = PatternFill("solid", fgColor="FFF2CC")
heatmap(row[0] for row in pairs_ws.iter_rows(min_row=2, min_col=5, max_col=5))
pairs_ws.auto_filter.ref = f"A1:G{len(p) + 1}"
pairs_ws.freeze_panes = "A2"
for col, width in zip("ABCDEFG", (max(map(len, names)) + 2, 9, max(map(len, names)) + 2, 9, 10, 10, 30)):
    pairs_ws.column_dimensions[col].width = width
done(f"{len(p):,} pairs, {(p['flag'] == 'redundant: keep one').sum()} flagged 'redundant: keep one'")

step("Read me sheet: method and color legend")
readme["A1"] = f"Correlation report: numerical variables more than {MIN_COMPLETENESS}% complete"
readme["A1"].font = Font(name=FONT, bold=True, size=14)
rows = [
    ("Source", f"data/master_listing.parquet, {len(df):,} listing episodes (one row per listing)"),
    ("Built by", f"eda/04_numerical_correlations.ipynb on {date.today():%Y-%m-%d}"),
    ("Variables", f"{n}: the {len(keep)} numerical variables more than {MIN_COMPLETENESS}% complete, "
                  f"minus {len(constant)} constant column(s) ({', '.join(constant) or 'none'})"),
    ("Method", f"{METHOD.title()} correlation; each pair uses the rows where both variables are present"),
    ("Values", "Computed in Python and written as values (no formulas); same numbers as data/correlations_numerical.csv"),
    ("", ""),
    ("Matrix", "Every variable vs every variable. Variable names are colored by role (see below)."),
    ("Pairs", "Each pair once, strongest first. Filter the 'flag' column on 'redundant: keep one'."),
    ("", ""),
    ("Cell colors", ""),
    ("-1.00", "strong negative: one goes up as the other goes down"),
    ("0.00", "no relationship"),
    ("+1.00", "strong positive: they go up together"),
    ("Bold", f"|correlation| >= {REDUNDANT}: the two variables carry the same information"),
    ("", ""),
    ("Name colors", ""),
    ("feature", "usable as X"),
    ("inferred", "usable as X (a platform inference available at prediction time)"),
    ("label", "a prediction target, not X"),
    ("leaky", "NEVER in X: this listing's own advertised price"),
    ("latent", "NEVER in X: simulator internals, not observable in production"),
]
for label, text in rows:
    readme.append([label, text])
for row in readme.iter_rows(min_row=2):
    row[0].font = Font(name=FONT, bold=True, color=ROLE_COLOR.get(row[0].value, "000000"))
    row[1].font = Font(name=FONT)
    row[1].alignment = Alignment(wrap_text=True, vertical="top")
for label, fill, text_color in (("-1.00", NEG, "FFFFFF"), ("0.00", MID, "000000"), ("+1.00", POS, "FFFFFF")):
    cell = next(c for c in readme["A"] if c.value == label)
    cell.fill = PatternFill("solid", fgColor=fill)
    cell.font = Font(name=FONT, bold=True, color=text_color)
    cell.alignment = Alignment(horizontal="center")
readme.column_dimensions["A"].width = 14
readme.column_dimensions["B"].width = 110

wb.active = 1   # open on the Matrix sheet
REPORT.parent.mkdir(parents=True, exist_ok=True)
wb.save(REPORT)
done(f"wrote {REPORT.relative_to(ROOT)}")


▶ Matrix sheet: every variable vs every variable


  ✓ 66 x 66 heatmap

▶ Pairs sheet: each pair once, strongest first


  ✓ 2,145 pairs, 15 flagged 'redundant: keep one'

▶ Read me sheet: method and color legend
  ✓ wrote eda/reports/correlations_numerical.xlsx
